In [11]:
import pandas as pd
import numpy as np
import pyodbc
import os

def get_sql_connection():
    """ایجاد اتصال به SQL Server"""
    return pyodbc.connect(
        'DRIVER={SQL Server};'
        'SERVER=MKZ-DSAS\\DSAS;'
        'DATABASE=DSAS;'
        'UID=datadriven;'
        'PWD=5Rdx@4Rfv1362'
    )

def fetch_and_transform_data():
    """
    خواندن داده از SQL Server، تغییر استراکچر و ذخیره در فایل اکسل
    """
    
    # مرحله 1: خواندن داده از SQL Server
    print("📥 مرحله 1: خواندن داده از SQL Server...")
    
    query = """
    SELECT TOP (10000000) [ID]
          ,[AssetID]
          ,[UnitID]
          ,[Value]
          ,[RecordTime]
          ,[RecordDate]
          ,[PersonelID]
          ,[OutofRange]
          ,[ValueType]
          ,[MobileID]
          ,[DateTime]
          ,[TimeStamp]
          ,[Job]
          ,[IsDeleted]
          ,[ShiftCode]
          ,[OnTime]
    FROM [DSAS].[PDA].[Periodic_Values]
    WHERE [UnitID]=11 AND
    ([AssetID]=9357 OR [AssetID]=9343 OR [AssetID]=8341 OR [AssetID]=8342 OR 
     [AssetID]=8343 OR [AssetID]=8344 OR [AssetID]=8346 OR [AssetID]=9286 OR 
     [AssetID]=9287 OR [AssetID]=9375)
    """
    
    try:
        conn = get_sql_connection()
        df = pd.read_sql(query, conn)
        conn.close()
        print(f"✅ داده با موفقیت از SQL خوانده شد. تعداد رکوردها: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا در خواندن داده از SQL: {e}")
        return None
    
    # مرحله 2: تغییر استراکچر داده (همانند کد اول)
    print("🔄 مرحله 2: تغییر استراکچر داده...")
    
    # استخراج AssetIDهای یکتا
    unique_ids = df['AssetID'].unique().tolist()
    print(f"🔢 تعداد AssetIDهای یکتا: {len(unique_ids)}")
    
    # ساخت دیتافریم خروجی با ستون‌های مورد نظر
    columns = ['TimeStamp'] + [f'AssetID_{uid}' for uid in unique_ids]
    output_df = pd.DataFrame(columns=columns)
    
    # پردازش تکراری تا خالی شدن دیتافریم اصلی
    row_count = 0
    while not df.empty and unique_ids:
        main_id = unique_ids[0]
        main_subset = df[df['AssetID'] == main_id]
        
        if main_subset.empty:
            unique_ids.pop(0)
            continue
        
        # گرفتن اولین ردیف از AssetID اصلی
        main_row = main_subset.iloc[0]
        main_ts = main_row['TimeStamp']
        main_value = main_row['Value']
        used_indices = [main_row.name]
        
        # پیدا کردن نزدیک‌ترین TimeStamp برای سایر AssetIDها
        row_data = {'TimeStamp': main_ts, f'AssetID_{main_id}': main_value}
        
        for other_id in unique_ids[1:]:
            subset = df[df['AssetID'] == other_id].copy()
            if subset.empty:
                row_data[f'AssetID_{other_id}'] = np.nan
                continue
            
            subset['ts_diff'] = np.abs(subset['TimeStamp'] - main_ts)
            close_rows = subset[subset['ts_diff'] <= 1800]
            
            if not close_rows.empty:
                closest_row = close_rows.sort_values('ts_diff').iloc[0]
                row_data[f'AssetID_{other_id}'] = closest_row['Value']
                used_indices.append(closest_row.name)
            else:
                row_data[f'AssetID_{other_id}'] = np.nan
        
        # حذف ردیف‌های استفاده‌شده
        df.drop(index=used_indices, inplace=True)
        
        # اضافه کردن ردیف جدید به خروجی
        output_df = pd.concat([output_df, pd.DataFrame([row_data])], ignore_index=True)
        row_count += 1
        
        # نمایش پیشرفت
        if row_count % 1000 == 0:
            print(f"   پردازش {row_count:,} رکورد...")
    
    print(f"✅ تعداد رکوردهای پردازش شده: {row_count:,}")
    
    # مرحله 3: حذف ردیف‌های دارای NaN
    print("🧹 مرحله 3: حذف ردیف‌های دارای مقادیر خالی...")
    asset_columns = [col for col in output_df.columns if col.startswith('AssetID_')]
    before_count = len(output_df)
    output_df = output_df.dropna(subset=asset_columns, how='any')
    after_count = len(output_df)
    print(f"   حذف {before_count - after_count:,} ردیف دارای مقادیر خالی")
    
    # مرحله 4: تبدیل TimeStamp به تاریخ و زمان
    print("📅 مرحله 4: تبدیل زمان‌ها...")
    output_df['date'] = pd.to_datetime(output_df['TimeStamp'], unit='s')
    output_df['RecordDate'] = output_df['date'].dt.date
    output_df['RecordTime'] = output_df['date'].dt.time
    
    # اضافه کردن ستون id
    output_df.insert(0, 'id', range(1, len(output_df) + 1))
    
    # حذف ستون TimeStamp
    output_df.drop(columns=['TimeStamp'], inplace=True)
    
    # مرتب‌سازی بر اساس تاریخ
    output_df.sort_values(by='date', inplace=True)
    
    # مرحله 5: ذخیره در فایل اکسل
    print("💾 مرحله 5: ذخیره در فایل اکسل...")
    
    # تعیین مسیر خروجی
    output_dir = r'second_stage_inputs\G11'
    output_file = os.path.join(output_dir, 'dsas_g11_lubrication_system_output.xlsx')
    
    # ایجاد پوشه در صورت عدم وجود
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        output_df.to_excel(output_file, index=False)
        print(f"✅ فایل با موفقیت ذخیره شد: {output_file}")
        print(f"📊 تعداد رکوردهای نهایی: {len(output_df):,}")
        print(f"📋 تعداد ستون‌ها: {len(output_df.columns)}")
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return None
    
    # نمایش اطلاعات آماری
    print("\n📊 اطلاعات آماری:")
    print(f"   بازه تاریخ: {output_df['date'].min()} تا {output_df['date'].max()}")
    print(f"   تعداد AssetIDها: {len(asset_columns)}")
    print("\n📋 نمونه‌ای از داده‌ها:")
    print(output_df.head(10))
    
    return output_df

# اجرای تابع اصلی
if __name__ == "__main__":
    print("="*60)
    print("🚀 شروع فرآیند استخراج و تبدیل داده")
    print("="*60)
    
    result_df = fetch_and_transform_data()
    
    if result_df is not None:
        print("\n" + "="*60)
        print("✅ فرآیند با موفقیت کامل شد!")
        print("="*60)
        
        # نمایش نام ستون‌ها
        print("\n📋 لیست ستون‌های فایل خروجی:")
        for i, col in enumerate(result_df.columns, 1):
            print(f"   {i:2d}. {col}")
    else:
        print("\n❌ فرآیند با شکست مواجه شد!")

🚀 شروع فرآیند استخراج و تبدیل داده
📥 مرحله 1: خواندن داده از SQL Server...


C:\Users\pishva_r\AppData\Local\Temp\ipykernel_44192\1300668866.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


✅ داده با موفقیت از SQL خوانده شد. تعداد رکوردها: 121,085
🔄 مرحله 2: تغییر استراکچر داده...
🔢 تعداد AssetIDهای یکتا: 10
   پردازش 1,000 رکورد...
   پردازش 2,000 رکورد...
   پردازش 3,000 رکورد...
   پردازش 4,000 رکورد...
   پردازش 5,000 رکورد...
   پردازش 6,000 رکورد...
   پردازش 7,000 رکورد...
   پردازش 8,000 رکورد...
   پردازش 9,000 رکورد...
   پردازش 10,000 رکورد...
   پردازش 11,000 رکورد...
   پردازش 12,000 رکورد...
   پردازش 13,000 رکورد...
   پردازش 14,000 رکورد...
   پردازش 15,000 رکورد...
   پردازش 16,000 رکورد...
   پردازش 17,000 رکورد...
✅ تعداد رکوردهای پردازش شده: 17,331
🧹 مرحله 3: حذف ردیف‌های دارای مقادیر خالی...
   حذف 10,422 ردیف دارای مقادیر خالی
📅 مرحله 4: تبدیل زمان‌ها...
💾 مرحله 5: ذخیره در فایل اکسل...
✅ فایل با موفقیت ذخیره شد: second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx
📊 تعداد رکوردهای نهایی: 6,909
📋 تعداد ستون‌ها: 14

📊 اطلاعات آماری:
   بازه تاریخ: 2021-03-20 17:37:26 تا 2026-06-28 20:30:34
   تعداد AssetIDها: 10

📋 نمونه‌ای از داده‌ها:
    